
### Notebook to collate all individual photometry files into one final file



In [ ]:

""" loading the relevant packages """

from glob import glob
import os
from datetime import date

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as tck
import matplotlib.cm as cm

import math

# setting figure style
plt.rcParams.update({
    "text.usetex" : False,
    "font.family" : "Arial", 
    "mathtext.fontset" : "stix",
    "font.size" : 14
})

fontsize = plt.rcParams["font.size"]


In [ ]:

""" defining some things ... """

transient_name = "SN2025sei"



### Defining some useful functions



In [2]:

def _mkdir(newdir):
    """ Creates the required directory for the output files. """
    """works the way a good mkdir should :)
        - already exists, silently complete
        - regular file in the way, raise an exception
        - parent directory(ies) does not exist, make them as well
    """
    if os.path.isdir(newdir):
        pass
    elif os.path.isfile(newdir):
        raise OSError("a file with the same name as the desired " \
                      "dir, '%s', already exists." % newdir)
    else:
        head, tail = os.path.split(newdir)
        if head and not os.path.isdir(head):
            _mkdir(head)
        #print "_mkdir %s" % repr(newdir)
        if tail:
            os.mkdir(newdir)
    return newdir


In [3]:

def fnu2ABmag(flux):
    # simple function to convert flux --> AB mag
    mag = -2.5 * np.log10(flux) + 23.90 # flux in micro-Jy
    return mag


def ABmag2fnu(mag):
    # simple function to convert AB mag --> flux
    flux = 10**((mag - 23.90) / (-2.5))
    return flux


def efnu2eABmag(flux, eflux):
    # simple function to convert flux error --> AB mag error
    emag = (2.5 / np.log(10)) * (eflux / flux)
    return emag


In [ ]:

""" defining Pan-STARRS filter information """

PS_filters           = ["g",     "r",     "i",     "z",     "y",     "w"]
PS_filter_eff_centre = [4810.16, 6155.47, 7503.03, 8668.36, 9613.60, 5980.70]
PS_filter_eff_width  = [1053.08, 1252.41, 1206.62, 997.72,  638.98,  2561.73]

PS_filter_df = pd.DataFrame(
    {
        "Filter" : PS_filters,
        "Eff_centre" : PS_filter_eff_centre,
        "Eff_width" : PS_filter_eff_width,
    }
)


In [ ]:

""" reading in -- and combining -- all individual photometry dataframes """

photometry_filenames = sorted(glob("Photometry/csv_files/*"))

mega_df = pd.DataFrame({})

for photometry_filename in photometry_filenames:

    if len(mega_df) == 0:
        mega_df = pd.read_csv(photometry_filename, sep=",")

    else:
        tmp_df = pd.read_csv(photometry_filename, sep=",")
        mega_df = pd.concat([mega_df, tmp_df])

mega_df.sort_values(by=["mjd", "filter"], ascending=True, inplace=True)

mega_df.reset_index(drop=True, inplace=True)

mega_df


,mjd,filter,filter_centre(AA),filter_width(AA),TotalExptime(s),Mag(AB),eMag(AB),Flux(uJy),eFlux(uJy),Mag_rounded(AB),eMag_rounded(AB)
0,60892.409343,i,7503.03,1206.62,180.0,18.114290,0.016579,206.197756,3.148646,18.11,0.02
1,60900.442922,i,7503.03,1206.62,360.0,18.125492,0.028004,204.081236,5.263798,18.13,0.03
2,60901.329090,g,4810.16,1053.08,120.0,18.154415,0.040537,198.716537,7.301230,18.15,0.04
3,60901.330700,r,6155.47,1252.41,120.0,18.199813,0.039698,190.578972,6.869330,18.20,0.04
4,60901.332280,i,7503.03,1206.62,120.0,18.218288,0.042408,187.363395,7.398990,18.22,0.04
...,...,...,...,...,...,...,...,...,...,...,...
103,60978.214070,r,6155.47,1252.41,120.0,31.401500,10436.063828,0.000999,9.611970,31.40,10436.06
104,60978.215630,i,7503.03,1206.62,120.0,20.995910,0.523012,14.508952,7.286240,21.00,0.52
105,60978.217200,z,8668.36,997.72,120.0,20.550895,0.418072,21.859598,8.596640,20.55,0.42
106,60978.218840,y,9613.60,638.98,120.0,19.811798,0.933819,43.179818,38.985130,19.81,0.93


In [ ]:

""" computing 3-sigma and 5-sigma upper limits (where necessary) """

mega_df["DetectionSignificance"] = mega_df["Flux(uJy)"] / mega_df["eFlux(uJy)"]

upper_lims_fivesigma_flux = []
upper_lims_threesigma_flux = []

for aa in range(len(mega_df)):
    tmp_df = mega_df.iloc[aa]

    sigma_cut = 3
    if tmp_df["DetectionSignificance"] < sigma_cut:
        flux_upper_lim = tmp_df["eFlux(uJy)"] * sigma_cut
    else:
        flux_upper_lim = "NaN"

    upper_lims_threesigma_flux.append(flux_upper_lim)


    sigma_cut = 5
    if tmp_df["DetectionSignificance"] < sigma_cut:
        flux_upper_lim = tmp_df["eFlux(uJy)"] * sigma_cut
    else:
        flux_upper_lim = "NaN"

    upper_lims_fivesigma_flux.append(flux_upper_lim)


mega_df["Flux_UL(3-sigma)"] = upper_lims_threesigma_flux
mega_df["Flux_UL(5-sigma)"] = upper_lims_fivesigma_flux

mega_df["Mag_UL(3-sigma)"] = fnu2ABmag(np.asarray(mega_df["Flux_UL(3-sigma)"]).astype(float))
mega_df["Mag_UL(5-sigma)"] = fnu2ABmag(np.asarray(mega_df["Flux_UL(5-sigma)"]).astype(float))

mega_df


,mjd,filter,filter_centre(AA),filter_width(AA),TotalExptime(s),Mag(AB),eMag(AB),Flux(uJy),eFlux(uJy),Mag_rounded(AB),eMag_rounded(AB),DetectionSignificance,Flux_UL(3-sigma),Flux_UL(5-sigma),Mag_UL(3-sigma),Mag_UL(5-sigma)
0,60892.409343,i,7503.03,1206.62,180.0,18.114290,0.016579,206.197756,3.148646,18.11,0.02,65.487756,NaN,NaN,NaN,NaN
1,60900.442922,i,7503.03,1206.62,360.0,18.125492,0.028004,204.081236,5.263798,18.13,0.03,38.770717,NaN,NaN,NaN,NaN
2,60901.329090,g,4810.16,1053.08,120.0,18.154415,0.040537,198.716537,7.301230,18.15,0.04,27.216858,NaN,NaN,NaN,NaN
3,60901.330700,r,6155.47,1252.41,120.0,18.199813,0.039698,190.578972,6.869330,18.20,0.04,27.743459,NaN,NaN,NaN,NaN
4,60901.332280,i,7503.03,1206.62,120.0,18.218288,0.042408,187.363395,7.398990,18.22,0.04,25.322834,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,60978.214070,r,6155.47,1252.41,120.0,31.401500,10436.063828,0.000999,9.611970,31.40,10436.06,0.000104,28.83591,48.05985,20.250166,19.695544
104,60978.215630,i,7503.03,1206.62,120.0,20.995910,0.523012,14.508952,7.286240,21.00,0.52,1.991281,21.85872,36.4312,20.550938,19.996316
105,60978.217200,z,8668.36,997.72,120.0,20.550895,0.418072,21.859598,8.596640,20.55,0.42,2.542807,25.78992,42.9832,20.371375,19.816753
106,60978.218840,y,9613.60,638.98,120.0,19.811798,0.933819,43.179818,38.985130,19.81,0.93,1.107597,116.95539,194.92565,18.729949,18.175328


In [ ]:

""" saving the final collated photometry file """

today = date.today()

mega_df.to_csv(f"{transient_name}_Pan-STARRS_photometry_{today}.csv", sep=",", index=False)

mega_df


,mjd,filter,filter_centre(AA),filter_width(AA),TotalExptime(s),Mag(AB),eMag(AB),Flux(uJy),eFlux(uJy),Mag_rounded(AB),eMag_rounded(AB),DetectionSignificance,Flux_UL(3-sigma),Flux_UL(5-sigma),Mag_UL(3-sigma),Mag_UL(5-sigma)
0,60892.409343,i,7503.03,1206.62,180.0,18.114290,0.016579,206.197756,3.148646,18.11,0.02,65.487756,NaN,NaN,NaN,NaN
1,60900.442922,i,7503.03,1206.62,360.0,18.125492,0.028004,204.081236,5.263798,18.13,0.03,38.770717,NaN,NaN,NaN,NaN
2,60901.329090,g,4810.16,1053.08,120.0,18.154415,0.040537,198.716537,7.301230,18.15,0.04,27.216858,NaN,NaN,NaN,NaN
3,60901.330700,r,6155.47,1252.41,120.0,18.199813,0.039698,190.578972,6.869330,18.20,0.04,27.743459,NaN,NaN,NaN,NaN
4,60901.332280,i,7503.03,1206.62,120.0,18.218288,0.042408,187.363395,7.398990,18.22,0.04,25.322834,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,60978.214070,r,6155.47,1252.41,120.0,31.401500,10436.063828,0.000999,9.611970,31.40,10436.06,0.000104,28.83591,48.05985,20.250166,19.695544
104,60978.215630,i,7503.03,1206.62,120.0,20.995910,0.523012,14.508952,7.286240,21.00,0.52,1.991281,21.85872,36.4312,20.550938,19.996316
105,60978.217200,z,8668.36,997.72,120.0,20.550895,0.418072,21.859598,8.596640,20.55,0.42,2.542807,25.78992,42.9832,20.371375,19.816753
106,60978.218840,y,9613.60,638.98,120.0,19.811798,0.933819,43.179818,38.985130,19.81,0.93,1.107597,116.95539,194.92565,18.729949,18.175328
